In [0]:
events = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/",
    header=True,
    inferSchema=True
)

events.count()

events.printSchema()


In [0]:
purchase_brands = (
    events_clean
    .filter(F.col("event_type") == "purchase")
    .groupBy("brand")
    .count()
    .orderBy(F.desc("count"))
)

purchase_brands.show(5)


In [0]:
events_clean = events_raw.filter(F.col("brand").isNotNull())

events_clean.printSchema()
events_clean.count()



In [0]:
purchase_brands = (
    events_clean
    .filter(F.col("event_type") == "purchase")
    .groupBy("brand")
    .count()
    .orderBy(F.desc("count"))
)

purchase_brands.show(5)


In [0]:
brand_revenue = (
    events_clean
    .filter(F.col("event_type") == "purchase")
    .groupBy("brand")
    .agg(F.sum("price").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

brand_revenue.show(5)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType


window_spec = (
    Window
    .partitionBy("user_id")
    .orderBy("event_time")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

events_running = events_clean.withColumn(
    "cumulative_events",
    F.count("*").over(window_spec)
)

events_running.select(
    "user_id", "event_time", "event_type", "cumulative_events"
).show(10)


In [0]:
purchase_window = (
    Window
    .partitionBy("user_id")
    .orderBy(F.desc("price"))
)

user_purchase_rank = (
    events_clean
    .filter(F.col("event_type") == "purchase")
    .withColumn("purchase_rank", F.dense_rank().over(purchase_window))
)

user_purchase_rank.select(
    "user_id", "price", "purchase_rank"
).show(10)


In [0]:
conversion = (
    events_clean
    .groupBy("category_code", "event_type")
    .count()
    .groupBy("category_code")
    .pivot("event_type")
    .sum("count")
    .withColumn(
        "conversion_rate",
        (F.col("purchase") / F.col("view")) * 100
    )
)

conversion.show(10)


In [0]:
def price_bucket(price):
    if price < 100:
        return "Low"
    elif price < 500:
        return "Medium"
    else:
        return "High"

price_udf = F.udf(price_bucket, StringType())

events_featured = events_clean.withColumn(
    "price_category",
    price_udf(F.col("price"))
)

events_featured.select("price", "price_category").show(10)

